In [1]:
!git clone https://github.com/Phani2603/DCP-ViT.git


Cloning into 'DCP-ViT'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 63 (delta 24), reused 26 (delta 23), pack-reused 23 (from 1)
Receiving objects: 100% (63/63), 171.93 KiB | 1.87 MiB/s, done.
Resolving deltas: 100% (25/25), done.


In [2]:
%cd DCP-ViT

/kaggle/working/DCP-ViT


In [3]:
!sed -i 's/==/>=/g' requirements.txt
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 2.8 MB/s eta 0:00:00


In [ ]:
import os

# Configure PyTorch memory allocation to avoid fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Install timm 0.6.13 which contains resolve_pretrained_cfg and supports older configurations
!pip install timm==0.6.13

# Restore original repository files to get rid of compatibility patches
!git checkout -- vision_transformer.py main.py engine.py linears.py

# Rename the conflicting local datasets.py module
if os.path.exists('datasets.py') and not os.path.exists('local_datasets.py'):
    os.rename('datasets.py', 'local_datasets.py')

# Replace local imports of 'datasets' with 'local_datasets'
!sed -i 's/import datasets/import local_datasets as datasets/g' main.py
!sed -i 's/from datasets import/from local_datasets import/g' main.py
!sed -i 's/import datasets/import local_datasets as datasets/g' engine.py
!sed -i 's/from datasets import/from local_datasets import/g' engine.py

# Inject explicit memory cleanup inside the main.py task loop to release GPU memory between tasks
with open('main.py', 'r') as f:
    main_content = f.read()

cleanup_code = """        # Reinitialising optimizer
        import gc
        import torch
        gc.collect()
        torch.cuda.empty_cache()
"""

main_content = main_content.replace("        # Reinitialising optimizer", cleanup_code)
with open('main.py', 'w') as f:
    f.write(main_content)

# Run the baseline with reduced batch size to stay within memory limits
!export TOKENIZERS_PARALLELISM=false && python -m main cifar100_convprompt \
  --num_tasks 10 \
  --data-path /content/drive/MyDrive/cifar100_data/ \
  --output_dir /content/drive/MyDrive/output_baseline \
  --dilation_rate 1 \
  --batch-size 64

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 5.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: timm
    Found existing installation: timm 1.0.26
    Uninstalling timm-1.0.26:
      Successfully uninstalled timm-1.0.26
Started main
Parser created:  ArgumentParser(prog='DualPrompt training and evaluation configs', usage=None, description=None, formatter_class=<class 'argparse.HelpFormatter'>, conflict_handler='error', add_help=True)
Getting config
Reached here
Reached here
Not using distributed mode
Train transforms:  Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.05, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ToTensor()
)
Test transforms:  Compose(
    Resize(size=256, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
)
100%|█████████████████████████████████████████| 169M/169M [09:31<00:00, 296kB/s]
NB CLasses:  100
Creating model: vit_base_patch16_